## Krok 0: Inicjalizacja Spark Session i Środowiska Windows

In [1]:
import os
import sys

current_dir = os.path.abspath("")
hadoop_dir = os.path.join(current_dir, "hadoop")
os.environ["HADOOP_HOME"] = hadoop_dir
os.environ["PATH"] = os.path.join(hadoop_dir, "bin") + os.pathsep + os.environ.get("PATH", "")

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ChicagoCrimesAnalysis") \
    .master("local[*]") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.hadoop.security.group.mapping", "org.apache.hadoop.security.ShellBasedUnixGroupsMapping") \
    .getOrCreate()

print("Spark Session utworzona pomyślnie!")
print(spark)

Spark Session utworzona pomyślnie!


## Krok 1: Wczytanie i Czyszczenie Danych

In [2]:
from pyspark.sql.functions import col

csv_path = "chicago_crimes_sample.csv"

df_raw = spark.read.option("header", "true") \
                   .option("multiLine", "true") \
                   .option("inferSchema", "true") \
                   .csv(csv_path)

print(f"Liczba wierszy przed czyszczeniem: {df_raw.count()}")

df_no_dups = df_raw.dropDuplicates()

key_columns = ["id", "case_number", "date", "primary_type"]
df_no_nulls = df_no_dups.dropna(subset=key_columns)

df_clean = df_no_nulls.filter(col("date").isNotNull())

print(f"Liczba wierszy po czyszczeniu: {df_clean.count()}")
df_clean.select("id", "case_number", "date", "primary_type", "location_description").show(5, truncate=False)

Liczba wierszy przed czyszczeniem: 50000
Liczba wierszy po czyszczeniu: 50000
+--------+-----------+-------------------+-----------------+----------------------+
|id      |case_number|date               |primary_type     |location_description  |
+--------+-----------+-------------------+-----------------+----------------------+
|14189334|JK244961   |2026-05-06 23:09:00|BATTERY          |CHA APARTMENT         |
|14189085|JK244797   |2026-05-06 18:45:00|CRIMINAL TRESPASS|RESTAURANT            |
|14189071|JK244784   |2026-05-06 16:15:00|ASSAULT          |SIDEWALK              |
|14188723|JK244326   |2026-05-06 13:00:00|CRIMINAL DAMAGE  |APARTMENT             |
|14188365|JK243801   |2026-05-06 04:30:00|CRIMINAL DAMAGE  |VEHICLE NON-COMMERCIAL|
+--------+-----------+-------------------+-----------------+----------------------+
only showing top 5 rows



## Krok 2: UDF (User Defined Function) i Klasyfikacja Pory Dnia

In [3]:
from pyspark.sql.functions import udf, hour, when
from pyspark.sql.types import StringType

def get_time_of_day(hr):
    if hr is None:
        return "nieznana"
    if 5 <= hr < 12:
        return "rano"
    elif 12 <= hr < 18:
        return "dzien"
    elif 18 <= hr < 22:
        return "wieczor"
    else:
        return "noc"

time_of_day_udf = udf(get_time_of_day, StringType())

try:
    df_with_tod = df_clean.withColumn("time_of_day", time_of_day_udf(hour(col("date"))))
    df_with_tod.select("date", "time_of_day").show(5)
except Exception as e:
    df_with_tod = df_clean.withColumn(
        "time_of_day",
        when((hour(col("date")) >= 5) & (hour(col("date")) < 12), "rano")
        .when((hour(col("date")) >= 12) & (hour(col("date")) < 18), "dzien")
        .when((hour(col("date")) >= 18) & (hour(col("date")) < 22), "wieczor")
        .otherwise("noc")
    )
    df_with_tod.select("date", "time_of_day").show(5)

Traceback (most recent call last):
  File "c:\_repos\python-lab\lab5\.venv\Lib\site-packages\pyspark\serializers.py", line 459, in dumps
    return cloudpickle.dumps(obj, pickle_protocol)
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\_repos\python-lab\lab5\.venv\Lib\site-packages\pyspark\cloudpickle\cloudpickle_fast.py", line 73, in dumps
    cp.dump(obj)
    ~~~~~~~^^^^^
  File "c:\_repos\python-lab\lab5\.venv\Lib\site-packages\pyspark\cloudpickle\cloudpickle_fast.py", line 632, in dump
    return Pickler.dump(self, obj)
           ~~~~~~~~~~~~^^^^^^^^^^^
  File "c:\_repos\python-lab\lab5\.venv\Lib\site-packages\pyspark\cloudpickle\cloudpickle_fast.py", line 737, in reducer_override
    return self._function_reduce(obj)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\_repos\python-lab\lab5\.venv\Lib\site-packages\pyspark\cloudpickle\cloudpickle_fast.py", line 592, in _function_reduce
    if _should_pickle_by_reference(obj):
       ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^
  Fi

+-------------------+-----------+
|               date|time_of_day|
+-------------------+-----------+
|2026-05-06 23:09:00|        noc|
|2026-05-06 18:45:00|    wieczor|
|2026-05-06 16:15:00|      dzien|
|2026-05-06 13:00:00|      dzien|
|2026-05-06 04:30:00|        noc|
+-------------------+-----------+
only showing top 5 rows



## Krok 3: Optymalizacja i Partycjonowanie

In [4]:
from pyspark.sql.functions import broadcast
import csv
import shutil

districts_dict_data = [
    (1, "Central-Loop"), (2, "Wentworth-South"), (3, "Grand Crossing-South"),
    (4, "South Chicago"), (5, "Calumet-Far South"), (6, "Gresham-South"),
    (7, "Englewood-South"), (8, "Chicago Lawn-Southwest"), (9, "Deering-Southwest"),
    (10, "Ogden-West"), (11, "Harrison-West"), (12, "Near West"),
    (14, "Shakespeare-Northwest"), (15, "Austin-West"), (16, "Jefferson Park-Northwest"),
    (17, "Albany Park-Northwest"), (18, "Near North"), (19, "Town Hall-North"),
    (20, "Lincoln-North"), (22, "Morgan Park-Far South"), (24, "Rogers Park-North"),
    (25, "Grand Central-Northwest")
]

districts_csv_path = "districts.csv"
with open(districts_csv_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["district", "district_zone_name"])
    writer.writerows(districts_dict_data)

df_districts = spark.read.option("header", "true") \
                         .option("inferSchema", "true") \
                         .csv(districts_csv_path)

df_optimized = df_with_tod.join(broadcast(df_districts), on="district", how="left")

df_optimized.cache()

cached_count = df_optimized.count()
print(f"Liczba rekordów w cache: {cached_count}")

output_parquet_path = "chicago_crimes_cleaned.parquet"

if os.path.exists(output_parquet_path):
    shutil.rmtree(output_parquet_path, ignore_errors=True)

df_optimized.write.mode("overwrite").partitionBy("year").parquet(output_parquet_path)
print("Dane zapisane do Parquet.")

Liczba rekordów w cache: 50000
Dane zapisane do Parquet.


## Krok 4: Analiza Statystyczna i Plany Zapytań

In [5]:
from pyspark.sql.functions import desc

crime_stats = df_optimized.groupBy("location_description", "time_of_day", "primary_type") \
                           .count() \
                           .orderBy(desc("count"))

print("Top 15 najczęstszych kombinacji lokalizacji, pory dnia i typu przestępstwa:")
crime_stats.show(15, truncate=False)

print("\n--- Plan zapytania dla powyższej agregacji (.explain(True)) ---")
crime_stats.explain(True)

Top 15 najczęstszych kombinacji lokalizacji, pory dnia i typu przestępstwa:
+--------------------+-----------+-------------------+-----+
|location_description|time_of_day|primary_type       |count|
+--------------------+-----------+-------------------+-----+
|STREET              |noc        |MOTOR VEHICLE THEFT|992  |
|APARTMENT           |noc        |BATTERY            |971  |
|STREET              |noc        |CRIMINAL DAMAGE    |795  |
|STREET              |wieczor    |MOTOR VEHICLE THEFT|771  |
|APARTMENT           |dzien      |BATTERY            |744  |
|APARTMENT           |rano       |BATTERY            |679  |
|STREET              |noc        |THEFT              |666  |
|STREET              |dzien      |MOTOR VEHICLE THEFT|627  |
|APARTMENT           |wieczor    |BATTERY            |625  |
|SMALL RETAIL STORE  |dzien      |THEFT              |591  |
|APARTMENT           |dzien      |THEFT              |589  |
|STREET              |rano       |THEFT              |568  |
|STREET  

## Krok 5: Model Uczenia Maszynowego (MLlib)

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

ml_df = df_optimized.select("primary_type", "location_description", "time_of_day", "arrest", "domestic", "district") \
                    .dropna()

ml_df = ml_df.withColumn("arrest_num", col("arrest").cast("double")) \
             .withColumn("domestic_num", col("domestic").cast("double")) \
             .withColumn("district_num", col("district").cast("double"))

indexer_location = StringIndexer(inputCol="location_description", outputCol="location_idx", handleInvalid="keep")
indexer_tod = StringIndexer(inputCol="time_of_day", outputCol="tod_idx", handleInvalid="keep")
indexer_label = StringIndexer(inputCol="primary_type", outputCol="label", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["location_idx", "tod_idx", "arrest_num", "domestic_num", "district_num"],
    outputCol="features"
)

rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=10, maxBins=150, seed=42)

pipeline = Pipeline(stages=[indexer_location, indexer_tod, indexer_label, assembler, rf])

train_data, test_data = ml_df.randomSplit([0.8, 0.2], seed=42)

print(f"Treningowe: {train_data.count()}, Testowe: {test_data.count()}")

model = pipeline.fit(train_data)
predictions = model.transform(test_data)
predictions.select("primary_type", "label", "prediction").show(10, truncate=False)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

Treningowe: 39815, Testowe: 9988
+------------+-----+----------+
|primary_type|label|prediction|
+------------+-----+----------+
|ASSAULT     |3.0  |0.0       |
|ASSAULT     |3.0  |0.0       |
|ASSAULT     |3.0  |1.0       |
|ASSAULT     |3.0  |1.0       |
|ASSAULT     |3.0  |0.0       |
|ASSAULT     |3.0  |0.0       |
|BATTERY     |1.0  |8.0       |
|BATTERY     |1.0  |1.0       |
|BATTERY     |1.0  |0.0       |
|BATTERY     |1.0  |1.0       |
+------------+-----+----------+
only showing top 10 rows


Test Accuracy: 33.10%


## Krok 6: Zatrzymanie sesji Spark

In [7]:
spark.stop()
print("Spark Session została poprawnie zatrzymana.")

try:
    os.remove("districts.csv")
    print("Tymczasowy plik districts.csv został usunięty.")
except Exception as e:
    pass

Spark Session została poprawnie zatrzymana.
Tymczasowy plik districts.csv został usunięty.
